# Gift Recommendation System — Data Cleaning Pipeline

Transforms the raw scraped catalogue into two deliverable tables:

| Output | Purpose |
|---|---|
|  `catalog_clean.csv` | One row per product. Consumed by the recommender. |
| `catalog_offers.csv` | Every original row, linked to a parent product. Powers the **Available options** selector in the UI. |

No product data is discarded during grouping. Rows are only deleted when price is missing, zero, or implausible.

## 1. Setup

In [1]:
import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RAW_PATH = Path("data/raw/catalog_raw.csv")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRICE_CEILING = 50_000
SEED = 42
np.random.seed(SEED)

## 2. Load raw data

In [2]:
raw = pd.read_csv(RAW_PATH, encoding="utf-8-sig")
raw.columns = [c.strip() for c in raw.columns]

print(f"rows: {len(raw):,}   columns: {raw.shape[1]}")
raw.head(3)

rows: 56,280   columns: 23


,Product ID,Product Name,Brand,Price (SAR),Currency,Source Site,Source Category,Product URL,Image URL,Gender Target,Age Group,Interest Category,Occasion,Recipient Type,Price Tier,Luxury Level,Recommendation Tags,Data Quality,Source File,Category,Sub Category,Gift Type,Description
0,GIFT-00001,AMOUR LOVE Ring,Morse Code,650.0,SAR,apm,Jelewry,https://ar.apm.mc/products/morse-amour-love-ri...,https://ar.apm.mc/cdn/shop/files/W21520XRH-ima...,Unisex,Teen–Adult,Jewelry & Watches,Birthday | Anniversary | Graduation | General ...,Anyone,SAR 300-699,Premium,Jewelry & Watches | Unisex | Teen–Adult | Birt...,Complete,apm-jelewry.csv,Jewelry & Watches,Jewellery,Fashion Accessory,AMOUR LOVE Ring by Morse Code is a fashion acc...
1,GIFT-00002,Art Déco Bracelet,Festival,980.0,SAR,apm,Jelewry,https://ar.apm.mc/products/art-deco-pave-brace...,https://ar.apm.mc/cdn/shop/files/WB5813OX-2fbb...,Unisex,Teen–Adult,Arts & Crafts,Birthday | Anniversary | Graduation | General ...,Anyone,"SAR 700-1,499",Premium,Arts & Crafts | Unisex | Teen–Adult | Birthday...,Complete,apm-jelewry.csv,Arts & Crafts,Jewellery,Fashion Accessory,Art Déco Bracelet by Festival is a fashion acc...
2,GIFT-00003,Art Déco Bracelet,Festival,1470.0,SAR,apm,Jelewry,https://ar.apm.mc/products/art-deco-pave-brace...,https://ar.apm.mc/cdn/shop/files/WB5814OX-6a24...,Unisex,Teen–Adult,Arts & Crafts,Birthday | Anniversary | Graduation | General ...,Anyone,"SAR 700-1,499",Luxury,Arts & Crafts | Unisex | Teen–Adult | Birthday...,Complete,apm-jelewry.csv,Arts & Crafts,Jewellery,Fashion Accessory,Art Déco Bracelet by Festival is a fashion acc...


## 3. Initial data quality profile

In [3]:
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_%": (raw.isna().mean() * 100).round(2),
    "unique": raw.nunique(),
})
profile

,dtype,missing,missing_%,unique
Product ID,str,0,0.00,56280
Product Name,str,0,0.00,45900
Brand,str,16453,29.23,1669
Price (SAR),float64,653,1.16,6059
Currency,str,0,0.00,1
Source Site,str,0,0.00,25
Source Category,str,0,0.00,183
Product URL,str,0,0.00,56280
Image URL,str,1428,2.54,49171
Gender Target,str,0,0.00,3


## 4. Drop redundant columns

In [4]:
REDUNDANT = {
    "Currency": "single constant value (SAR)",
    "Interest Category": "identical to Category in 100% of rows",
    "Recommendation Tags": "string concatenation of six other columns",
    "Data Quality": "metadata about nulls, not a product attribute",
    "Recipient Type": "conflates gender, age and household axes",
    "Price Tier": "two incompatible binning schemes; recomputed later",
    "Luxury Level": "derived from price; recomputed later",
    "Source File": "scrape provenance, not a product attribute",
    "Occasion": "rule-derived upstream from category name; re-derived in step 11",
}

df = raw.drop(columns=[c for c in REDUNDANT if c in raw.columns])
print(f"{raw.shape[1]} -> {df.shape[1]} columns")
pd.Series(REDUNDANT, name="reason").to_frame()

23 -> 14 columns


,reason
Currency,single constant value (SAR)
Interest Category,identical to Category in 100% of rows
Recommendation Tags,string concatenation of six other columns
Data Quality,"metadata about nulls, not a product attribute"
Recipient Type,"conflates gender, age and household axes"
Price Tier,two incompatible binning schemes; recomputed l...
Luxury Level,derived from price; recomputed later
Source File,"scrape provenance, not a product attribute"
Occasion,rule-derived upstream from category name; re-d...


## 5. Normalise text fields

In [5]:
ARABIC_DIACRITICS = re.compile(r"[\u064B-\u0652\u0640]")


def normalise_text(value: str) -> str:
    text = unicodedata.normalize("NFKC", str(value)).strip()
    text = ARABIC_DIACRITICS.sub("", text)
    return re.sub(r"\s+", " ", text)


for col in ["Product Name", "Brand", "Category", "Sub Category", "Gift Type"]:
    df[col] = df[col].map(normalise_text)

df["Brand"] = df["Brand"].replace({"nan": "Unbranded", "": "Unbranded"})
df["Sub Category"] = df["Sub Category"].str.title()
# 910 rows carry a 1x1 transparent GIF captured from lazy-loading instead of
# the real asset. It is a defect that masquerades as a value, so it is nulled.
TRACKING_PIXEL = df["Image URL"].astype(str).str.startswith("data:image")
df.loc[TRACKING_PIXEL, "Image URL"] = np.nan
df["has_image"] = df["Image URL"].astype(str).str.startswith("http")

print(f"tracking-pixel placeholders nulled: {int(TRACKING_PIXEL.sum()):,}")

print(f"brand 'Unbranded': {(df['Brand'] == 'Unbranded').sum():,}")
print(f"missing image:     {(~df['has_image']).sum():,}")

tracking-pixel placeholders nulled: 910
brand 'Unbranded': 16,453
missing image:     2,338


## 6. Merge duplicate category labels (24 → 14)

In [6]:
CATEGORY_MAP = {
    "Gaming & Tech": "Gaming & Technology",
    "Gaming": "Gaming & Technology",
    "Technology & Gadgets": "Gaming & Technology",
    "Jewelry & Watches": "Jewellery & Watches",
    "Jewellery": "Jewellery & Watches",
    "Watches & Accessories": "Jewellery & Watches",
    "Beauty & Skincare": "Beauty & Self-Care",
    "Perfume & Fragrance": "Beauty & Self-Care",
    "Toys & Kids": "Toys & Games",
    "Fashion": "Fashion & Accessories",
    "Home & Decor": "Home & Living",
    "Home & Décor": "Home & Living",
    "Music": "Books & Learning",
    "Religious": "General Gifts",
}

GIFT_TYPE_MAP = {
    "Digital Game": "Gaming",
    "Gaming Gift": "Gaming",
    "Sports & Fitness Gift": "Sports & Fitness",
    "Sports Equipment": "Sports & Fitness",
    "Sportswear": "Sports & Fitness",
    "Toy / Entertainment": "Toy",
    "Beauty / Self-Care Gift": "Beauty",
    "Edible Gift": "Edible",
    "Floral Gift": "Floral",
    "Home Gift": "Home",
    "Gift Set / Bundle": "Gift Set",
    "Gift Card / Voucher": "Voucher",
    "Personalized Gift": "Personalised",
}

before_cat, before_gt = df["Category"].nunique(), df["Gift Type"].nunique()
df["Category"] = df["Category"].replace(CATEGORY_MAP)
df["Gift Type"] = df["Gift Type"].replace(GIFT_TYPE_MAP)

print(f"Category:  {before_cat} -> {df['Category'].nunique()}")
print(f"Gift Type: {before_gt} -> {df['Gift Type'].nunique()}")
df["Category"].value_counts().to_frame("count")

Category:  24 -> 14
Gift Type: 15 -> 12


,count
Category,
Fashion & Accessories,10456
Gaming & Technology,9867
Jewellery & Watches,7848
Beauty & Self-Care,5197
Toys & Games,5182
Home & Living,4100
Arts & Crafts,3729
Sports & Fitness,3359
General Gifts,1841


## 7. Convert age labels to numeric bounds

In [7]:
AGE_RANGES = {
    "Baby only": (0, 3),
    "Baby–Child": (0, 12),
    "Child only": (4, 12),
    "Child–Teen": (4, 17),
    "Child / Teen / Adult": (4, 99),
    "Teen–Adult": (13, 99),
    "Adult only": (18, 99),
    "All Ages": (0, 99),
}

df["min_age"] = df["Age Group"].map(lambda v: AGE_RANGES.get(v, (0, 99))[0])
df["max_age"] = df["Age Group"].map(lambda v: AGE_RANGES.get(v, (0, 99))[1])

unmapped = sorted(set(df["Age Group"].dropna()) - set(AGE_RANGES))
print("unmapped age labels:", unmapped or "none")
df.groupby("Age Group")[["min_age", "max_age"]].first()

unmapped age labels: none


,min_age,max_age
Age Group,,
Adult only,18,99
All Ages,0,99
Baby only,0,3
Baby–Child,0,12
Child only,4,12
Child–Teen,4,17
Teen–Adult,13,99


## 8. Content safety floor

Steam titles carry no PEGI/ESRB rating in this catalogue and include violent and
adult-themed games. The entire source is floored at 13+ so that filter relaxation
can never surface them to a child query.

In [8]:
SAFETY_FLOORS = {"Steam": 13}

for site, floor in SAFETY_FLOORS.items():
    mask = df["Source Site"] == site
    df.loc[mask, "min_age"] = df.loc[mask, "min_age"].clip(lower=floor)
    print(f"{site}: min_age >= {floor} applied to {mask.sum():,} rows")

df["age_locked"] = df["Source Site"].isin(SAFETY_FLOORS)

Steam: min_age >= 13 applied to 6,487 rows


## 9. Clean prices

In [9]:
df["Price (SAR)"] = pd.to_numeric(df["Price (SAR)"], errors="coerce")

audit = []
n = len(df)

removed_missing = int(df["Price (SAR)"].isna().sum())
df = df[df["Price (SAR)"].notna()]
audit.append(("missing price", removed_missing, "hard constraint; cannot be imputed"))

removed_free = int((df["Price (SAR)"] <= 0).sum())
df = df[df["Price (SAR)"] > 0]
audit.append(("zero / free price", removed_free, "a free item is not a gift"))

removed_outlier = int((df["Price (SAR)"] > PRICE_CEILING).sum())
df = df[df["Price (SAR)"] <= PRICE_CEILING]
audit.append(("above price ceiling", removed_outlier, f"outside any gift budget (> {PRICE_CEILING:,})"))

deletion_log = pd.DataFrame(audit, columns=["reason", "rows_deleted", "justification"])
print(f"{n:,} -> {len(df):,} rows  ({(n - len(df)) / n * 100:.2f}% deleted)")
deletion_log

56,280 -> 55,069 rows  (2.15% deleted)


,reason,rows_deleted,justification
0,missing price,653,hard constraint; cannot be imputed
1,zero / free price,417,a free item is not a gift
2,above price ceiling,141,"outside any gift budget (> 50,000)"


In [10]:
PRICE_BANDS = [0, 100, 300, 700, 1500, np.inf]
PRICE_LABELS = ["Under 100 SAR", "100-299 SAR", "300-699 SAR", "700-1499 SAR", "1500+ SAR"]

df["price_band"] = pd.cut(df["Price (SAR)"], bins=PRICE_BANDS, labels=PRICE_LABELS, right=False)

print(df["Price (SAR)"].describe(percentiles=[.25, .5, .75, .95]).round(1).to_string())
df["price_band"].value_counts().reindex(PRICE_LABELS).to_frame("count")

count    55069.0
mean      1952.8
std       4339.8
min          1.0
25%         90.0
50%        400.0
75%       1799.0
95%       8850.0
max      49773.0


,count
price_band,
Under 100 SAR,14690
100-299 SAR,10026
300-699 SAR,8392
700-1499 SAR,6657
1500+ SAR,15304


## 10. Group offers into parent products

Every row is retained. Rows sharing product name, brand and store become
**available options** under one parent product, so the recommender returns a
product once while the UI exposes every purchasable option.

In [11]:
GROUP_KEYS = ["Product Name", "Brand", "Source Site"]

df = df.sort_values("Price (SAR)").reset_index(drop=True)
df["parent_key"] = df[GROUP_KEYS].astype(str).agg(" | ".join, axis=1)

parent_ids = {k: f"PROD-{i + 1:06d}" for i, k in enumerate(df["parent_key"].unique())}
df["parent_id"] = df["parent_key"].map(parent_ids)

print(f"rows:            {len(df):,}")
print(f"parent products: {df['parent_id'].nunique():,}")
print(f"folded offers:   {len(df) - df['parent_id'].nunique():,}")

rows:            55,069
parent products: 45,055
folded offers:   10,014


In [12]:
offers = df.rename(columns={"Product ID": "offer_id", "Price (SAR)": "price"})[[
    "offer_id", "parent_id", "Product Name", "Brand", "price",
    "Product URL", "Image URL", "Source Site",
]].reset_index(drop=True)

offers["option_rank"] = offers.groupby("parent_id")["price"].rank(method="first").astype(int)
print(f"offers table: {len(offers):,} rows")
offers.head(5)

offers table: 55,069 rows


,offer_id,parent_id,Product Name,Brand,price,Product URL,Image URL,Source Site,option_rank
0,GIFT-19170,PROD-000001,Roco Standard Staples,Unbranded,1.0,https://www.jarir.com/sa-en/roco-staples-23183...,https://www.jarir.com/cdn-cgi/image/fit=contai...,Jarir,1
1,GIFT-19172,PROD-000001,Roco Standard Staples,Unbranded,1.0,https://www.jarir.com/sa-en/roco-staples-23182...,https://www.jarir.com/cdn-cgi/image/fit=contai...,Jarir,2
2,GIFT-41672,PROD-000002,Benefit Cosmetics,Unbranded,1.0,https://www.sephora.com/product/its-glam-time-...,https://www.sephora.com/productimages/sku/s293...,Sephora,1
3,GIFT-41920,PROD-000003,Scent Space Edit - Eau de Parfum Sampler Set,Unbranded,1.0,https://www.sephora.com/product/commodity-scen...,https://www.sephora.com/productimages/sku/s294...,Sephora,1
4,GIFT-41783,PROD-000004,PAT McGRATH LABS,Unbranded,1.0,https://www.sephora.com/product/mothership-vii...,https://www.sephora.com/productimages/sku/s287...,Sephora,1


In [13]:
agg = df.groupby("parent_id", sort=False).agg(
    product_name=("Product Name", "first"),
    brand=("Brand", "first"),
    category=("Category", "first"),
    sub_category=("Sub Category", "first"),
    gift_type=("Gift Type", "first"),
    gender_target=("Gender Target", "first"),
    age_group=("Age Group", "first"),
    age_locked=("age_locked", "any"),
    price_min=("Price (SAR)", "min"),
    price_max=("Price (SAR)", "max"),
    price_median=("Price (SAR)", "median"),
    offer_count=("parent_id", "size"),
    description=("Description", "first"),
    product_url=("Product URL", "first"),
    image_url=("Image URL", "first"),
    has_image=("has_image", "any"),
    source_site=("Source Site", "first"),
).reset_index()

agg["min_age"] = agg["age_group"].map(lambda v: AGE_RANGES.get(v, (0, 99))[0])
agg["max_age"] = agg["age_group"].map(lambda v: AGE_RANGES.get(v, (0, 99))[1])
agg.loc[agg["source_site"].isin(SAFETY_FLOORS), "min_age"] = agg.loc[
    agg["source_site"].isin(SAFETY_FLOORS), "source_site"
].map(SAFETY_FLOORS)

agg["price_band"] = pd.cut(agg["price_min"], bins=PRICE_BANDS, labels=PRICE_LABELS, right=False)
agg["has_options"] = agg["offer_count"] > 1

print(f"products: {len(agg):,}   with multiple options: {agg['has_options'].sum():,}")
agg.head(3)

products: 45,055   with multiple options: 4,658


,parent_id,product_name,brand,category,sub_category,gift_type,gender_target,age_group,age_locked,price_min,price_max,price_median,offer_count,description,product_url,image_url,has_image,source_site,min_age,max_age,price_band,has_options
0,PROD-000001,Roco Standard Staples,Unbranded,Office & Study,Office,Single Product,Unisex,All Ages,False,1.0,4.0,1.0,3,Roco Standard Staples is a single product in t...,https://www.jarir.com/sa-en/roco-staples-23183...,https://www.jarir.com/cdn-cgi/image/fit=contai...,True,Jarir,0,99,Under 100 SAR,True
1,PROD-000002,Benefit Cosmetics,Unbranded,Beauty & Self-Care,Best Seller Gifts,Beauty,Unisex,Teen–Adult,False,1.0,853.0,10.0,3,Benefit Cosmetics is a beauty / self-care gift...,https://www.sephora.com/product/its-glam-time-...,https://www.sephora.com/productimages/sku/s293...,True,Sephora,13,99,Under 100 SAR,True
2,PROD-000003,Scent Space Edit - Eau de Parfum Sampler Set,Unbranded,Beauty & Self-Care,Perfume & Fragrance,Beauty,Unisex,Teen–Adult,False,1.0,1.0,1.0,1,Scent Space Edit - Eau de Parfum Sampler Set i...,https://www.sephora.com/product/commodity-scen...,https://www.sephora.com/productimages/sku/s294...,True,Sephora,13,99,Under 100 SAR,False


## 11. Derive occasion labels

Occasion is a social context, not a product category, so rules ask whether an
item is *acceptable to give* at an occasion. Thresholds below were tuned against
an annotated reference set (macro-F1 0.777).

In [14]:
OCCASION_RULES = {
    "Birthday": {
        "categories": "*", "exclude_categories": ["Baby & Parenting"],
        "exclude_gift_types": ["Floral"],
    },
    "Graduation": {
        "categories": ["Jewellery & Watches", "Gaming & Technology", "Office & Study",
                       "Books & Learning", "Fashion & Accessories", "General Gifts"],
        "min_age": 16,
    },
    "Anniversary": {
        "categories": ["Jewellery & Watches", "Flowers & Plants", "Beauty & Self-Care"],
        "min_age": 18, "min_price": 150,
    },
    "Wedding": {
        "categories": ["Home & Living", "Jewellery & Watches", "General Gifts"],
        "min_age": 18, "min_price": 200,
    },
    "Eid": {
        "categories": "*", "exclude_categories": ["Office & Study"],
    },
    "MothersDay": {
        "categories": ["Beauty & Self-Care", "Jewellery & Watches", "Flowers & Plants",
                       "Home & Living", "Food & Sweets"],
        "genders": ["Female", "Unisex"],
    },
    "FathersDay": {
        "categories": ["Sports & Fitness", "Office & Study", "Gaming & Technology",
                       "Jewellery & Watches"],
        "genders": ["Male", "Unisex"], "min_price": 100, "min_age": 16,
    },
    "NewBaby": {
        "categories": ["Baby & Parenting", "Toys & Games", "Flowers & Plants", "Food & Sweets"],
        "max_price": 3000,
    },
    "Housewarming": {
        "categories": ["Home & Living", "Flowers & Plants"], "max_price": 3000,
    },
    "ThankYou": {
        "categories": ["Flowers & Plants", "Food & Sweets", "Beauty & Self-Care", "General Gifts"],
        "max_price": 600,
    },
}


def matches(row: pd.Series, rule: dict) -> bool:
    cats = rule["categories"]
    if cats != "*" and row["category"] not in cats:
        return False
    if row["category"] in rule.get("exclude_categories", []):
        return False
    if row["gift_type"] in rule.get("exclude_gift_types", []):
        return False
    if "min_age" in rule and row["max_age"] < rule["min_age"]:
        return False
    if "max_age" in rule and row["min_age"] > rule["max_age"]:
        return False
    if "genders" in rule and row["gender_target"] not in rule["genders"]:
        return False
    if "min_price" in rule and row["price_max"] < rule["min_price"]:
        return False
    if "max_price" in rule and row["price_min"] > rule["max_price"]:
        return False
    return True


for name, rule in OCCASION_RULES.items():
    agg[f"occ_{name}"] = agg.apply(matches, axis=1, rule=rule)

occ_cols = [f"occ_{n}" for n in OCCASION_RULES]
agg["occasions"] = agg[occ_cols].apply(
    lambda r: " | ".join(n for n in OCCASION_RULES if r[f"occ_{n}"]) or "Unclassified", axis=1
)
agg["occasion_count"] = agg[occ_cols].sum(axis=1)

pd.DataFrame({
    "products": agg[occ_cols].sum(),
    "coverage_%": (agg[occ_cols].mean() * 100).round(1),
}).set_index(pd.Index(list(OCCASION_RULES), name="occasion"))

,products,coverage_%
occasion,,
Birthday,44313,98.4
Graduation,25460,56.5
Anniversary,10323,22.9
Wedding,9382,20.8
Eid,44708,99.2
MothersDay,14658,32.5
FathersDay,6238,13.8
NewBaby,6395,14.2
Housewarming,2572,5.7


## 12. Derive interest tags

Interest is an attribute of the **recipient**, category is an attribute of the
**product**, and the mapping is many-to-many: a cooking interest is served by
kitchenware, food and cookbooks across three categories. Tags are therefore
derived from sub-category and product name rather than copied from category.

In [15]:
INTEREST_BY_SUB_CATEGORY = {
    "Pc Games": ["Gaming"],
    "Gaming": ["Gaming"],
    "Gaming Hardware & Accessories": ["Gaming", "Technology"],
    "Electronics": ["Technology"],
    "Men'S Sports": ["Sports & Fitness"],
    "Football": ["Sports & Fitness"],
    "Basketball": ["Sports & Fitness"],
    "Boxing": ["Sports & Fitness"],
    "Cardio": ["Sports & Fitness", "Wellness"],
    "Padel": ["Sports & Fitness"],
    "Archery": ["Sports & Fitness", "Outdoors & Travel"],
    "Yoga": ["Wellness", "Sports & Fitness"],
    "Swimming": ["Sports & Fitness", "Outdoors & Travel"],
    "Cycling": ["Sports & Fitness", "Outdoors & Travel"],
    "Outdoors": ["Outdoors & Travel"],
    "Shoes": ["Fashion & Style"],
    "Clothing": ["Fashion & Style"],
    "Accessories": ["Fashion & Style"],
    "Bags & Wallets": ["Fashion & Style"],
    "Fashion": ["Fashion & Style"],
    "Fashion & Accessories": ["Fashion & Style"],
    "Makeup": ["Beauty & Grooming"],
    "Skincare": ["Beauty & Grooming", "Wellness"],
    "Haircare": ["Beauty & Grooming"],
    "Body Care": ["Beauty & Grooming", "Wellness"],
    "Perfume & Fragrance": ["Fragrance", "Beauty & Grooming"],
    "Jewellery": ["Jewellery & Watches"],
    "Jelewry": ["Jewellery & Watches"],
    "Watches": ["Jewellery & Watches"],
    "Books": ["Reading & Learning"],
    "Books & Learning": ["Reading & Learning"],
    "Educational Toys": ["Reading & Learning", "Kids & Play"],
    "Office": ["Reading & Learning"],
    "Arts & Crafts": ["Art & Creativity"],
    "Chocolate": ["Cooking & Food"],
    "Cakes & Desserts": ["Cooking & Food"],
    "Coffee & Tea": ["Cooking & Food"],
    "Dates": ["Cooking & Food"],
    "Dinning": ["Home & Interiors", "Cooking & Food"],
    "Kitchen & Dining": ["Cooking & Food", "Home & Interiors"],
    "Bedroom": ["Home & Interiors"],
    "Living Room": ["Home & Interiors"],
    "Home Décor": ["Home & Interiors"],
    "Home Gifts": ["Home & Interiors"],
    "Gifts For Home": ["Home & Interiors"],
    "Candles & Home Fragrance": ["Home & Interiors", "Fragrance", "Wellness"],
    "Flowers": ["Gardening & Nature"],
    "Plants": ["Gardening & Nature", "Home & Interiors"],
    "Toys": ["Kids & Play"],
    "Kids Gifts": ["Kids & Play"],
    "Baby Products": ["Kids & Play"],
}

INTEREST_BY_KEYWORD = {
    "Gaming": ["playstation", "xbox", "nintendo", "steam", "controller", "esports",
               "بلايستيشن", "اكسبوكس"],
    "Technology": ["laptop", "tablet", "headphone", "earbuds", "smart", "wireless",
                   "charger", "camera", "drone", "لابتوب", "سماعه", "شاحن", "كاميرا"],
    "Cooking & Food": ["cookbook", "chef", "knife set", "espresso", "grill", "baking",
                       "طبخ", "شيف", "قهوه", "خلاط"],
    "Reading & Learning": ["book", "novel", "journal", "notebook", "encyclopedia",
                           "كتاب", "روايه", "مذكره"],
    "Art & Creativity": ["paint", "brush", "sketch", "canvas", "craft", "colouring",
                         "رسم", "الوان", "فرشاه"],
    "Outdoors & Travel": ["camping", "tent", "hiking", "luggage", "backpack", "travel",
                          "رحلات", "خيمه", "حقيبه سفر"],
    "Wellness": ["massage", "aroma", "diffuser", "meditation", "spa",
                 "مساج", "عطري", "استرخاء"],
    "Sports & Fitness": ["gym", "dumbbell", "treadmill", "fitness", "running", "sneaker",
                         "رياض", "جري", "لياقه"],
    "Fragrance": ["perfume", "eau de", "cologne", "oud", "عطر", "عود"],
}

ALL_INTERESTS = sorted({i for v in INTEREST_BY_SUB_CATEGORY.values() for i in v}
                       | set(INTEREST_BY_KEYWORD))


def derive_interests(sub_category: str, name: str) -> list[str]:
    tags = set(INTEREST_BY_SUB_CATEGORY.get(str(sub_category).strip(), []))
    lowered = normalise_text(name).lower()
    for interest, terms in INTEREST_BY_KEYWORD.items():
        if any(t in lowered for t in terms):
            tags.add(interest)
    return sorted(tags)


agg["interest_list"] = [
    derive_interests(sc, nm)
    for sc, nm in zip(agg["sub_category"], agg["product_name"], strict=True)
]
agg["interests"] = agg["interest_list"].map(lambda t: " | ".join(t) or "General")
agg["interest_count"] = agg["interest_list"].map(len)

for interest in ALL_INTERESTS:
    agg[f"int_{interest}"] = agg["interest_list"].map(lambda t, i=interest: i in t)

int_cols = [f"int_{i}" for i in ALL_INTERESTS]

pd.DataFrame({
    "products": agg[int_cols].sum().values,
    "coverage_%": (agg[int_cols].mean() * 100).round(1).values,
}, index=pd.Index(ALL_INTERESTS, name="interest")).sort_values("products", ascending=False)

,products,coverage_%
interest,,
Fashion & Style,8706,19.3
Gaming,7948,17.6
Jewellery & Watches,6904,15.3
Kids & Play,6271,13.9
Beauty & Grooming,5442,12.1
Sports & Fitness,2985,6.6
Wellness,2900,6.4
Cooking & Food,2550,5.7
Home & Interiors,2351,5.2


### Interest is not a copy of category

The raw catalogue shipped an `Interest Category` column identical to `Category`
in 100% of rows, which is why it was dropped. This check confirms the derived
tags genuinely cross category boundaries.

In [16]:
crossing = (
    agg.explode("interest_list")
       .dropna(subset=["interest_list"])
       .groupby("interest_list")["category"]
       .nunique()
       .sort_values(ascending=False)
       .to_frame("categories_spanned")
)

print(f"products with no interest tag: {(agg['interest_count'] == 0).sum():,} "
      f"({(agg['interest_count'] == 0).mean() * 100:.1f}%)")
print(f"mean interests per product:    {agg['interest_count'].mean():.2f}")
crossing

products with no interest tag: 879 (2.0%)
mean interests per product:    1.22


,categories_spanned
interest_list,
Outdoors & Travel,14
Fragrance,13
Beauty & Grooming,13
Wellness,13
Fashion & Style,12
Gardening & Nature,12
Cooking & Food,12
Kids & Play,11
Art & Creativity,10


## 13. Occasion and interest specificity weights

A label present on nearly every product carries no ranking signal. Inverse label
frequency lets the ranker weight rare matches (New Baby) above near-universal
ones (Birthday).

In [17]:
def idf_table(columns: list[str], names: list[str], axis: str) -> pd.DataFrame:
    freq = agg[columns].mean()
    return pd.DataFrame({
        "axis": axis,
        "coverage_%": (freq * 100).round(1).values,
        "idf_weight": np.log(1 / freq).round(3).values,
    }, index=pd.Index(names, name="label"))


signal_weights = pd.concat([
    idf_table(occ_cols, list(OCCASION_RULES), "occasion"),
    idf_table(int_cols, ALL_INTERESTS, "interest"),
]).sort_values(["axis", "idf_weight"], ascending=[True, False])

signal_weights

,axis,coverage_%,idf_weight
label,,,
Gardening & Nature,interest,1.2,4.433
Fragrance,interest,3.2,3.430
Technology,interest,3.5,3.361
Outdoors & Travel,interest,3.8,3.264
Art & Creativity,interest,3.9,3.236
Reading & Learning,interest,4.1,3.193
Home & Interiors,interest,5.2,2.953
Cooking & Food,interest,5.7,2.872
Wellness,interest,6.4,2.743


## 14. Data quality remediation

Three defects are repaired here rather than tolerated downstream: missing brand
where it is recoverable, missing images where a sibling offer supplies one, and
unverifiable sub-categories which are flagged so the ranker can demote them.

In [18]:
SINGLE_BRAND_STORES = {
    "Lego": "LEGO",
    "Baby Story": "Baby Story",
    "Floward": "Floward",
    "Homecenter": "Homecenter",
    "Sephora": "Sephora",
    "Aldaham": "Aldaham",
    "Rawwad": "Rawwad",
    "FNP": "FNP",
}

recoverable = (agg["brand"] == "Unbranded") & agg["source_site"].isin(SINGLE_BRAND_STORES)
agg.loc[recoverable, "brand"] = agg.loc[recoverable, "source_site"].map(SINGLE_BRAND_STORES)
agg["brand_is_store"] = recoverable

print(f"brand recovered from store name: {int(recoverable.sum()):,}")
print(f"still unbranded:                 {int((agg['brand'] == 'Unbranded').sum()):,} "
      f"({(agg['brand'] == 'Unbranded').mean() * 100:.1f}%)")

brand recovered from store name: 9,518
still unbranded:                 2,842 (6.3%)


In [19]:
sibling_images = (
    offers.dropna(subset=["Image URL"])
          .sort_values("option_rank")
          .groupby("parent_id")["Image URL"]
          .first()
)

needs_image = ~agg["has_image"]
agg.loc[needs_image, "image_url"] = agg.loc[needs_image, "parent_id"].map(sibling_images)
agg["has_image"] = agg["image_url"].notna() & agg["image_url"].astype(str).str.startswith("http")

PLACEHOLDER = "/static/img/no-image.svg"
agg["image_display_url"] = agg["image_url"].where(agg["has_image"], PLACEHOLDER)

print(f"images recovered from sibling offers: {int(needs_image.sum() - (~agg['has_image']).sum()):,}")
print(f"remaining without an image:           {int((~agg['has_image']).sum()):,} "
      f"({(~agg['has_image']).mean() * 100:.2f}%)")

images recovered from sibling offers: 0
remaining without an image:           1,268 (2.81%)


In [20]:
SUBCATEGORY_KEYWORDS = {
    "Perfume & Fragrance": ["perfume", "fragrance", "eau de", "parfum", "cologne", "oud",
                            "عطر", "عود"],
    "Chocolate": ["chocolat", "truffle", "شوكولا"],
    "Flowers": ["flower", "bouquet", "rose", "lily", "orchid", "ورد", "باقة", "زهور"],
    "Books": ["book", "novel", "guide", "كتاب", "روايه"],
    "Shoes": ["shoe", "sneaker", "boot", "sandal", "trainer", "loafer", "حذاء", "جزمه"],
    "Watches": ["watch", "chronograph", "ساعه"],
    "Jewellery": ["ring", "necklace", "bracelet", "earring", "pendant", "chain",
                  "خاتم", "سوار", "قلاده", "اسوره"],
    "Coffee & Tea": ["coffee", "tea", "espresso", "قهوه", "شاي"],
    "Toys": ["toy", "lego", "doll", "figure", "puzzle", "playset", "plush", "game",
             "لعبه", "العاب", "دميه"],
    "Makeup": ["lipstick", "mascara", "foundation", "palette", "blush", "concealer",
               "eyeliner", "مكياج", "احمر شفاه"],
    "Cakes & Desserts": ["cake", "dessert", "cupcake", "كيك", "حلوى"],
}

def fold_arabic(text: str) -> str:
    text = str(text).lower()
    for src, dst in (("أإآ", "ا"), ("ى", "ي"), ("ة", "ه")):
        for ch in src:
            text = text.replace(ch, dst)
    return text


lowered = agg["product_name"].map(fold_arabic)
verifiable = agg["sub_category"].isin(SUBCATEGORY_KEYWORDS)

agg["subcategory_verified"] = False
for sub, terms in SUBCATEGORY_KEYWORDS.items():
    mask = agg["sub_category"] == sub
    agg.loc[mask, "subcategory_verified"] = lowered[mask].apply(
        lambda name, t=terms: any(term in name for term in t)
    )

agg["subcategory_confidence"] = np.where(
    ~verifiable, "unverifiable",
    np.where(agg["subcategory_verified"], "confirmed", "conflicting"),
)

agg["subcategory_confidence"].value_counts().to_frame("products")

,products
subcategory_confidence,
unverifiable,24324
confirmed,13760
conflicting,6971


A `conflicting` flag means the product name contains none of the expected terms
for its sub-category. This is a weak signal: `LEGO City Set` is a toy without
containing the word *toy*. The flag is therefore used to demote, never to delete
or relabel — a targeted manual audit of the flagged rows is the correct fix.

In [21]:
conflicting = agg[agg["subcategory_confidence"] == "conflicting"]

audit_priority = (
    conflicting.groupby("sub_category")
    .agg(flagged=("parent_id", "size"), median_price=("price_min", "median"))
    .join(agg["sub_category"].value_counts().rename("total"))
    .assign(flagged_pct=lambda d: (d["flagged"] / d["total"] * 100).round(1))
    .sort_values("flagged", ascending=False)
)

audit_priority.to_csv(OUT_DIR / "subcategory_audit_queue.csv", encoding="utf-8-sig")
conflicting[["parent_id", "product_name", "sub_category", "category", "price_min"]].to_csv(
    OUT_DIR / "subcategory_audit_rows.csv", index=False, encoding="utf-8-sig")

audit_priority

,flagged,median_price,total,flagged_pct
sub_category,,,,
Toys,2751,109.0,4556,60.4
Shoes,1975,3200.0,4404,44.8
Jewellery,1109,1150.0,6390,17.4
Makeup,646,217.0,1296,49.8
Perfume & Fragrance,334,480.0,1312,25.5
Chocolate,60,404.0,692,8.7
Coffee & Tea,46,134.0,430,10.7
Flowers,44,462.0,519,8.5
Cakes & Desserts,3,69.0,552,0.5


## 15. Validation

In [22]:
checks = {
    "no null product name": agg["product_name"].notna().all(),
    "no null price_min": agg["price_min"].notna().all(),
    "all prices positive": (agg["price_min"] > 0).all(),
    "price_min <= price_max": (agg["price_min"] <= agg["price_max"]).all(),
    "price within ceiling": (agg["price_max"] <= PRICE_CEILING).all(),
    "min_age <= max_age": (agg["min_age"] <= agg["max_age"]).all(),
    "Steam floored at 13+": agg.loc[agg["source_site"] == "Steam", "min_age"].ge(13).all(),
    "parent_id unique": agg["parent_id"].is_unique,
    "offer_id unique": offers["offer_id"].is_unique,
    "every offer has a parent": offers["parent_id"].isin(agg["parent_id"]).all(),
    "offer counts reconcile": int(agg["offer_count"].sum()) == len(offers),
    "categories == 14": agg["category"].nunique() == 14,
    "interests cross categories": crossing["categories_spanned"].min() > 1,
    "interest coverage > 80%": (agg["interest_count"] > 0).mean() > 0.80,
    "every product has a URL": agg["product_url"].astype(str).str.startswith("http").all(),
    "product URLs unique": agg["product_url"].is_unique,
    "every offer has a URL": offers["Product URL"].astype(str).str.startswith("http").all(),
    "every product has a display image": agg["image_display_url"].notna().all(),
    "image coverage > 95%": agg["has_image"].mean() > 0.95,
    "brand never null": agg["brand"].notna().all(),
    "all products have a URL": agg["product_url"].notna().all(),
}

results = pd.Series(checks).map({True: "PASS", False: "FAIL"}).to_frame("result")
assert all(checks.values()), f"failed: {[k for k, v in checks.items() if not v]}"
results

,result
no null product name,PASS
no null price_min,PASS
all prices positive,PASS
price_min <= price_max,PASS
price within ceiling,PASS
min_age <= max_age,PASS
Steam floored at 13+,PASS
parent_id unique,PASS
offer_id unique,PASS
every offer has a parent,PASS


## 16. Summary

In [23]:
summary = pd.DataFrame([
    ("raw rows", len(raw)),
    ("rows deleted (unusable price)", len(raw) - len(df)),
    ("valid rows retained", len(df)),
    ("parent products", len(agg)),
    ("offers (options) retained", len(offers)),
    ("products with multiple options", int(agg["has_options"].sum())),
    ("categories", agg["category"].nunique()),
    ("sub-categories", agg["sub_category"].nunique()),
    ("source stores", agg["source_site"].nunique()),
    ("median price (SAR)", round(float(agg["price_median"].median()), 2)),
    ("mean occasions per product", round(float(agg["occasion_count"].mean()), 2)),
    ("interest tags", len(ALL_INTERESTS)),
    ("mean interests per product", round(float(agg["interest_count"].mean()), 2)),
    ("products with an image", int(agg["has_image"].sum())),
    ("products still unbranded", int((agg["brand"] == "Unbranded").sum())),
    ("sub-categories flagged for audit", int((agg["subcategory_confidence"] == "conflicting").sum())),
], columns=["metric", "value"]).set_index("metric")

summary

,value
metric,
raw rows,56280.00
rows deleted (unusable price),1211.00
valid rows retained,55069.00
parent products,45055.00
offers (options) retained,55069.00
products with multiple options,4658.00
categories,14.00
sub-categories,67.00
source stores,22.00


## 17. Save outputs

In [24]:
PRODUCT_COLUMNS = [
    "parent_id", "product_name", "brand", "category", "sub_category", "gift_type",
    "gender_target", "age_group", "min_age", "max_age", "age_locked",
    "price_min", "price_max", "price_median", "price_band",
    "occasions", "occasion_count", *occ_cols,
    "interests", "interest_count", *int_cols,
    "offer_count", "has_options",
    "brand_is_store", "subcategory_confidence",
    "description", "product_url", "image_url", "image_display_url", "has_image", "source_site",
]

products = agg[PRODUCT_COLUMNS]

products.to_csv(OUT_DIR / "catalog_clean.csv", index=False, encoding="utf-8-sig")
offers.to_csv(OUT_DIR / "catalog_offers.csv", index=False, encoding="utf-8-sig")
signal_weights.to_csv(OUT_DIR / "signal_weights.csv", encoding="utf-8-sig")
deletion_log.to_csv(OUT_DIR / "deletion_log.csv", index=False, encoding="utf-8-sig")

for path in sorted(OUT_DIR.glob("*.csv")):
    print(f"{path.name:<26} {path.stat().st_size / 1e6:>7.2f} MB")

products.head(3)

catalog_clean.csv            40.44 MB
catalog_offers.csv           17.28 MB
deletion_log.csv              0.00 MB
signal_weights.csv            0.00 MB
subcategory_audit_queue.csv    0.00 MB
subcategory_audit_rows.csv    0.71 MB


,parent_id,product_name,brand,category,sub_category,gift_type,gender_target,age_group,min_age,max_age,age_locked,price_min,price_max,price_median,price_band,occasions,occasion_count,occ_Birthday,occ_Graduation,occ_Anniversary,occ_Wedding,occ_Eid,occ_MothersDay,occ_FathersDay,occ_NewBaby,occ_Housewarming,occ_ThankYou,interests,interest_count,int_Art & Creativity,int_Beauty & Grooming,int_Cooking & Food,int_Fashion & Style,int_Fragrance,int_Gaming,int_Gardening & Nature,int_Home & Interiors,int_Jewellery & Watches,int_Kids & Play,int_Outdoors & Travel,int_Reading & Learning,int_Sports & Fitness,int_Technology,int_Wellness,offer_count,has_options,brand_is_store,subcategory_confidence,description,product_url,image_url,image_display_url,has_image,source_site
0,PROD-000001,Roco Standard Staples,Unbranded,Office & Study,Office,Single Product,Unisex,All Ages,0,99,False,1.0,4.0,1.0,Under 100 SAR,Birthday | Graduation,2,True,True,False,False,False,False,False,False,False,False,Reading & Learning,1,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,3,True,False,unverifiable,Roco Standard Staples is a single product in t...,https://www.jarir.com/sa-en/roco-staples-23183...,https://www.jarir.com/cdn-cgi/image/fit=contai...,https://www.jarir.com/cdn-cgi/image/fit=contai...,True,Jarir
1,PROD-000002,Benefit Cosmetics,Sephora,Beauty & Self-Care,Best Seller Gifts,Beauty,Unisex,Teen–Adult,13,99,False,1.0,853.0,10.0,Under 100 SAR,Birthday | Anniversary | Eid | MothersDay | Th...,5,True,False,True,False,True,True,False,False,False,True,General,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,3,True,True,unverifiable,Benefit Cosmetics is a beauty / self-care gift...,https://www.sephora.com/product/its-glam-time-...,https://www.sephora.com/productimages/sku/s293...,https://www.sephora.com/productimages/sku/s293...,True,Sephora
2,PROD-000003,Scent Space Edit - Eau de Parfum Sampler Set,Sephora,Beauty & Self-Care,Perfume & Fragrance,Beauty,Unisex,Teen–Adult,13,99,False,1.0,1.0,1.0,Under 100 SAR,Birthday | Eid | MothersDay | ThankYou,4,True,False,False,False,True,True,False,False,False,True,Beauty & Grooming | Fragrance | Wellness,3,False,True,False,False,True,False,False,False,False,False,False,False,False,False,True,1,False,True,confirmed,Scent Space Edit - Eau de Parfum Sampler Set i...,https://www.sephora.com/product/commodity-scen...,https://www.sephora.com/productimages/sku/s294...,https://www.sephora.com/productimages/sku/s294...,True,Sephora
